In [2]:
import numpy as np
import pandas as pd

# =========================
# 1. Chargement
# =========================
X = np.load("X_lstm.npy")
y = np.load("y_lstm.npy")

indices = ["NDVI", "EVI", "NDMI", "NDWI"]

N, T, F = X.shape

print("Shape X :", X.shape)

# =========================
# 2. Extraction des features temporelles
# =========================
rows = []

for i in range(N):
    sample = X[i]

    # enlever padding (lignes constantes)
    valid_mask = ~(np.all(sample == sample[:, [0]], axis=1))
    valid_sample = sample[valid_mask]

    if len(valid_sample) == 0:
        continue

    row = {
        "label": y[i],
        "n_valid_dates": len(valid_sample)
    }

    # features pour chaque indice
    for f, name in enumerate(indices):
        ts = valid_sample[:, f]

        row[f"{name}_mean"] = np.mean(ts)
        row[f"{name}_max"] = np.max(ts)
        row[f"{name}_min"] = np.min(ts)
        row[f"{name}_std"] = np.std(ts)
        row[f"{name}_amplitude"] = np.max(ts) - np.min(ts)

    rows.append(row)

df = pd.DataFrame(rows)

print(df.head())
print("Shape final :", df.shape)

# =========================
# 3. Sauvegarde
# =========================
df.to_csv("temporal_features_full.csv", index=False)

Shape X : (112002, 30, 4)
   label  n_valid_dates  NDVI_mean  NDVI_max  NDVI_min  NDVI_std  \
0     22              3   0.161892  0.349968 -0.208113  0.261645   
1     23              1   0.266662  0.266662  0.266662  0.000000   
2     23              1   0.290532  0.290532  0.290532  0.000000   
3     19             30   0.147223  0.290317 -0.195326  0.127997   
4     12             30   0.193948  0.315356 -0.171867  0.125034   

   NDVI_amplitude  EVI_mean   EVI_max   EVI_min  ...  NDMI_mean  NDMI_max  \
0        0.558081  1.540953  2.929635 -0.070129  ...   0.064555  0.082499   
1        0.000000  1.406540  1.406540  1.406540  ...  -0.071024 -0.071024   
2        0.000000  1.148803  1.148803  1.148803  ...   0.070997  0.070997   
3        0.485643  0.638115  1.210040 -0.089916  ...  -0.022460  0.120958   
4        0.487223  0.852888  1.733629 -0.117421  ...  -0.006492  0.154806   

   NDMI_min  NDMI_std  NDMI_amplitude  NDWI_mean  NDWI_max  NDWI_min  \
0  0.043311  0.016168        0